# 🤖 Prophet Training – Dự báo doanh thu
**Mô hình:** Facebook Prophet (additive time-series)

**Công thức:** y(t) = g(t) + s(t) + h(t) + ε(t)
- g(t): trend (xu hướng tổng thể)
- s(t): seasonality (tuần, tháng, năm)
- h(t): holiday effects (ngày lễ Việt Nam)
- ε(t): noise

**Input:** prophet_dataset.csv (từ notebook 01_EDA)

**Output:** models/prophet_revenue.pkl

In [ ]:
!pip install prophet pandas scikit-learn numpy matplotlib joblib -q
print('✅ Cài đặt hoàn tất')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.figsize'] = (14, 5)
print('✅ Import hoàn tất')

## 1. Tải dữ liệu

In [ ]:
# Tải dataset từ bước EDA (hoặc load trực tiếp từ DB)
df = pd.read_csv('prophet_dataset.csv', parse_dates=['ds'])
df = df.sort_values('ds').reset_index(drop=True)

print(f'📦 Dataset: {len(df)} ngày')
print(f'   Từ: {df["ds"].min().date()} → {df["ds"].max().date()}')
print(f'   y min: {df["y"].min():,.0f} | y max: {df["y"].max():,.0f} | y mean: {df["y"].mean():,.0f}')
df.head()

## 2. Train/Test split

In [ ]:
# Dùng 80% cuối để train, 20% cuối để test
split_idx = int(len(df) * 0.8)
df_train = df.iloc[:split_idx].copy()
df_test  = df.iloc[split_idx:].copy()

print(f'Train: {len(df_train)} ngày ({df_train["ds"].min().date()} → {df_train["ds"].max().date()})')
print(f'Test : {len(df_test)}  ngày ({df_test["ds"].min().date()} → {df_test["ds"].max().date()})')

plt.figure(figsize=(14, 4))
plt.plot(df_train['ds'], df_train['y'], label='Train', color='steelblue')
plt.plot(df_test['ds'],  df_test['y'],  label='Test',  color='orange')
plt.legend()
plt.title('Train / Test split')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Định nghĩa ngày lễ Việt Nam

In [ ]:
# Ngày lễ Việt Nam có ảnh hưởng đến doanh thu
vn_holidays = pd.DataFrame({
    'holiday': [
        'Tết Nguyên Đán', 'Tết Nguyên Đán', 'Tết Nguyên Đán',
        'Giỗ Tổ Hùng Vương', 'Giải phóng miền Nam', 'Quốc tế Lao động',
        'Quốc khánh', 'Black Friday', '12/12 Sale', '11/11 Sale',
    ],
    'ds': pd.to_datetime([
        '2024-02-10', '2024-02-11', '2024-02-12',  # Tết 2024
        '2024-04-18',  # Giỗ Tổ Hùng Vương
        '2024-04-30',  # Giải phóng
        '2024-05-01',  # Lao động
        '2024-09-02',  # Quốc khánh
        '2024-11-29',  # Black Friday
        '2024-12-12',  # 12/12
        '2024-11-11',  # 11/11
    ]),
    'lower_window': [-2, -1, 0, 0, 0, 0, 0, -1, 0, 0],
    'upper_window': [7,   6,  5, 1, 1, 1, 1,  1, 1, 1],
})

print('✅ Ngày lễ Việt Nam đã định nghĩa:')
print(vn_holidays[['holiday','ds']].to_string(index=False))

## 4. Train mô hình Prophet

In [ ]:
model = Prophet(
    # Trend
    changepoint_prior_scale=0.05,    # Độ linh hoạt của trend (0.01-0.5)
    changepoint_range=0.8,           # 80% dữ liệu đầu để phát hiện changepoint

    # Seasonality
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,         # Dữ liệu ngày → không cần daily
    seasonality_mode='multiplicative',  # Phù hợp dữ liệu bán lẻ có growth

    # Holidays
    holidays=vn_holidays,
    holidays_prior_scale=10.0,       # Mức ảnh hưởng của ngày lễ

    # Uncertainty
    interval_width=0.95,             # 95% confidence interval
)

# Thêm seasonality theo tháng (monthly pattern cho TMĐT VN)
model.add_seasonality(name='monthly', period=30.5, fourier_order=5)

print('⏳ Đang train...')
model.fit(df_train)
print('✅ Train hoàn tất!')

## 5. Dự báo và đánh giá

In [ ]:
# Tạo future dataframe gồm cả test set + 30 ngày tương lai
horizon_days  = len(df_test) + 30
future        = model.make_future_dataframe(periods=horizon_days)
forecast      = model.predict(future)

# Vẽ dự báo
fig1 = model.plot(forecast, figsize=(14, 5))
plt.title('Prophet Forecast – Doanh thu thuần', fontweight='bold')
plt.xlabel('Ngày')
plt.ylabel('Doanh thu (VNĐ)')
plt.tight_layout()
plt.savefig('prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# Vẽ components
fig2 = model.plot_components(forecast, figsize=(14, 10))
plt.tight_layout()
plt.savefig('prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tính MAE, RMSE, MAPE trên test set
forecast_test = forecast[forecast['ds'].isin(df_test['ds'])][['ds','yhat','yhat_lower','yhat_upper']]
df_eval = df_test.merge(forecast_test, on='ds')

mae  = mean_absolute_error(df_eval['y'], df_eval['yhat'])
rmse = np.sqrt(mean_squared_error(df_eval['y'], df_eval['yhat']))
mape = np.mean(np.abs((df_eval['y'] - df_eval['yhat']) / df_eval['y'].replace(0, np.nan))) * 100

print('=== Kết quả đánh giá mô hình trên tập Test ===')
print(f'  MAE  : {mae:>15,.0f} VNĐ')
print(f'  RMSE : {rmse:>15,.0f} VNĐ')
print(f'  MAPE : {mape:>14.2f}%')

# Visualize actual vs predicted
plt.figure(figsize=(14, 5))
plt.plot(df_eval['ds'], df_eval['y'],    label='Thực tế',  color='steelblue', linewidth=2)
plt.plot(df_eval['ds'], df_eval['yhat'], label='Dự báo',   color='orange',    linewidth=2, linestyle='--')
plt.fill_between(df_eval['ds'], df_eval['yhat_lower'], df_eval['yhat_upper'],
                 alpha=0.2, color='orange', label='95% CI')
plt.title(f'Thực tế vs Dự báo – Test Set (MAPE={mape:.2f}%)', fontweight='bold')
plt.ylabel('Doanh thu (VNĐ)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('prophet_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cross-validation

In [ ]:
# Cross-validation: cutoff mỗi 30 ngày, horizon 14 ngày
# (Điều chỉnh initial nếu dataset nhỏ hơn 365 ngày)
df_cv = cross_validation(
    model,
    initial='180 days',
    period='30 days',
    horizon='14 days',
    parallel='processes',
)
df_perf = performance_metrics(df_cv)
print('=== Cross-validation Metrics ===')
print(df_perf[['horizon','mae','rmse','mape']].to_string(index=False))

fig = plot_cross_validation_metric(df_cv, metric='mape')
plt.title('MAPE theo horizon (Cross-validation)', fontweight='bold')
plt.tight_layout()
plt.savefig('prophet_cv_mape.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export model

In [ ]:
os.makedirs('models', exist_ok=True)

# Lưu model Prophet bằng joblib
MODEL_PATH = 'models/prophet_revenue.pkl'
joblib.dump(model, MODEL_PATH)
print(f'✅ Model đã lưu tại: {MODEL_PATH}')

# Lưu metadata để FastAPI load
import json
metadata = {
    'model_type':    'Prophet',
    'trained_at':    pd.Timestamp.now().isoformat(),
    'train_rows':    len(df_train),
    'train_from':    str(df_train['ds'].min().date()),
    'train_to':      str(df_train['ds'].max().date()),
    'mae':           round(mae, 0),
    'rmse':          round(rmse, 0),
    'mape_pct':      round(mape, 2),
    'horizon_days':  30,
    'seasonality_mode': 'multiplicative',
}
with open('models/prophet_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('✅ Metadata đã lưu tại: models/prophet_metadata.json')
print(json.dumps(metadata, indent=2, ensure_ascii=False))
print('\n📌 Bước tiếp theo: Copy models/ vào src/ai-service/models/ để FastAPI load')